# Día 19 — Limpieza de datos y Joins en pandas

**Dataset:** Tienda Tech — `productos.csv` + `ventas.csv`  
**Objetivo:** detectar y tratar problemas en datos reales antes de analizarlos.

---

In [9]:
# import pandas as pd → carga la librería pandas y la renombra como pd
# a partir de aquí escribimos pd.algo en vez de pandas.algo
import pandas as pd

# pd.read_csv('ruta') → lee el archivo CSV y lo convierte en un DataFrame (tabla en memoria)
# '../data/day19/productos.csv' → .. sube un nivel desde notebooks/, luego entra en data/day19/
# el DataFrame resultante se guarda en la variable productos
productos = pd.read_csv('../data/day19/productos.csv')
ventas    = pd.read_csv('../data/day19/ventas.csv')

# .shape → propiedad del DataFrame que devuelve una tupla (número de filas, número de columnas)
# print('texto:', valor) → imprime el texto literal seguido del valor de la variable
print('Productos:', productos.shape)
print('Ventas:   ', ventas.shape)
print(productos.head())
print(ventas.head())

Productos: (50, 6)
Ventas:    (50, 6)
   producto_id                nombre    categoria     marca  precio  stock
0          101    Laptop ProBook 450      Laptops        HP  899.99   15.0
1          102  Monitor UltraWide 34    Monitores        LG  449.99    8.0
2          103   Teclado Mecánico K3     Teclados  Keychron  129.99    NaN
3          104  Ratón Inalámbrico MX      Ratones  Logitech   79.99   22.0
4          105   Auriculares WH-1000  Auriculares      Sony  299.99   11.0
   venta_id  producto_id       fecha  cantidad     total       cliente
0         1          101  15/01/2024         2  1799.98€  Carlos López
1         2          103  22/01/2024         1   129.99€    Ana García
2         3          115  30/01/2024         3   839.97€           NaN
3         4          107  05/02/2024         1   399.99€  María Torres
4         5          102  12/02/2024         2   899.98€    Pedro Ruiz


---
## Concepto 1 — Detectar nulos: `df.isnull().sum()`

Los datos reales siempre tienen valores vacíos. Antes de analizar hay que saber exactamente dónde están y cuántos.

- `df.isnull()` devuelve un DataFrame de booleanos: `True` donde hay nulo
- `.sum()` suma los `True` por columna (True = 1, False = 0)
- Dividir entre `len(df)` da el porcentaje de nulos

**Ejercicio 1:** detectar nulos en `productos` y mostrar su porcentaje por columna.

In [2]:
# productos.isnull() → crea un DataFrame idéntico pero con True donde hay NaN y False donde hay dato
# .sum() → suma los True por columna (True vale 1, False vale 0)
# el resultado es una Serie donde el índice es el nombre de la columna y el valor es el conteo de nulos
nulos = productos.isnull().sum()

# nulos / len(productos) → divide cada conteo entre el total de filas
# len(productos) devuelve el número de filas (equivale a productos.shape[0])
# * 100 → convierte la proporción (0.08) en porcentaje (8.0)
# .round(2) → redondea el resultado a 2 decimales
pct   = (nulos / len(productos) * 100).round(2)

# pd.DataFrame({...}) → construye un DataFrame nuevo desde un diccionario
# cada clave del diccionario se convierte en el nombre de una columna
# cada valor (nulos, pct) es la Serie que rellena esa columna fila por fila
# el índice del DataFrame resultante son los nombres de las columnas de productos
resumen = pd.DataFrame({'nulos': nulos, 'porcentaje_%': pct})

# resumen['nulos'] > 0 → crea una Serie booleana: True donde el conteo de nulos es mayor que 0
# resumen[...] → usa esa Serie como máscara y filtra: solo muestra las filas donde es True
print(resumen[resumen['nulos'] > 0])

           nulos  porcentaje_%
categoria      1           2.0
marca          1           2.0
precio         2           4.0
stock          4           8.0


---
## Concepto 2 — Rellenar o eliminar nulos: `fillna()` vs `dropna()`

No todos los nulos se tratan igual. La decisión depende de lo que representa la columna:

- `df['col'].fillna(valor)` → sustituye el nulo por un valor concreto
- `df.dropna(subset=['col'])` → elimina filas donde esa columna es nula

**Regla:** usa `fillna` cuando el nulo tiene un significado lógico (sin stock = 0).  
Usa `dropna` cuando la fila no puede analizarse sin ese dato (sin precio = no analizable).

**Ejercicio 2:** en `productos`, rellenar `stock` nulo con 0 y eliminar filas donde `precio` es nulo.

In [3]:
# productos['stock'] → accede a la columna 'stock' del DataFrame (devuelve una Serie)
# .fillna(0) → recorre cada valor de esa Serie y reemplaza los NaN por 0
# productos['stock'] = → sobreescribe la columna entera con el resultado (ya sin NaN)
productos['stock'] = productos['stock'].fillna(0)

# len(productos) cuenta las filas actuales; lo guardamos antes de eliminar
# así podemos calcular después cuántas filas desaparecieron
filas_antes = len(productos)

# .dropna(subset=['precio']) → elimina las filas donde la columna 'precio' es NaN
# subset=['precio'] acota la búsqueda a esa columna concreta
# sin subset= eliminaría cualquier fila que tenga algún NaN en cualquier columna
# productos = → sobreescribe la variable con el DataFrame resultante (ahora tiene menos filas)
productos = productos.dropna(subset=['precio'])

# filas_antes - len(productos) → resta filas actuales a las que había antes = filas eliminadas
print('Filas eliminadas por precio nulo:', filas_antes - len(productos))
print('Nulos restantes en stock y precio:')
# productos[['stock', 'precio']] → selecciona solo esas dos columnas (lista dentro de corchetes)
# .isnull().sum() → cuenta nulos en cada una de esas dos columnas
print(productos[['stock', 'precio']].isnull().sum())

Filas eliminadas por precio nulo: 2
Nulos restantes en stock y precio:
stock     0
precio    0
dtype: int64


---
## Concepto 3 — Eliminar duplicados: `drop_duplicates()`

Una fila duplicada es idéntica a otra que ya existe en el DataFrame. Sin eliminarlas, las sumas y conteos estarán inflados.

- `df.duplicated()` → Serie booleana: `True` donde la fila es un duplicado
- `df.duplicated(keep=False)` → marca **todas** las ocurrencias, no solo la segunda
- `df.drop_duplicates()` → elimina duplicados y conserva la primera ocurrencia

**Ejercicio 3:** detectar y eliminar filas duplicadas en `productos`.

In [4]:
# productos.duplicated() → devuelve una Serie booleana donde True significa
# que esa fila ya apareció antes en el DataFrame (es una copia)
# .sum() cuenta cuántos True hay → número total de filas duplicadas
print('Filas duplicadas:', productos.duplicated().sum())

# keep=False → marca TODAS las apariciones del duplicado como True (original Y copia)
# sin keep=False solo la segunda aparición sería True; no veríamos el original
# productos[...] → filtra y devuelve solo las filas marcadas como True
# [['producto_id', 'nombre']] → de esas filas, muestra solo esas dos columnas
# .sort_values('producto_id') → ordena por producto_id para ver los duplicados juntos
print('Detalle de duplicados:')
print(productos[productos.duplicated(keep=False)][['producto_id', 'nombre']].sort_values('producto_id'))

# drop_duplicates() → recorre el DataFrame y elimina las filas repetidas
# conserva la primera aparición de cada fila y descarta las siguientes
# productos = → sobreescribe la variable con el resultado limpio
productos = productos.drop_duplicates()
print('Shape final:', productos.shape)

Filas duplicadas: 3
Detalle de duplicados:
    producto_id                nombre
0           101    Laptop ProBook 450
47          101    Laptop ProBook 450
1           102  Monitor UltraWide 34
48          102  Monitor UltraWide 34
2           103   Teclado Mecánico K3
49          103   Teclado Mecánico K3
Shape final: (45, 6)


---
## Concepto 4 — Verificar tipos: `df.dtypes`

Una columna con números guardados como texto (`object`) no se puede sumar ni calcular. Hay que convertirla primero.

- `df.dtypes` → muestra el tipo de cada columna
- `pd.to_datetime(df['col'], format='%d/%m/%Y')` → convierte string a fecha
- `df['col'].str.replace('€', '').astype(float)` → quita el símbolo y convierte a número

**Ejercicio 4:** corregir `fecha` (string `dd/mm/yyyy`) y `total` (string con `€`) en `ventas`.

In [5]:
# .dtypes → propiedad del DataFrame que devuelve el tipo de cada columna
# object = texto (string), int64 = entero, float64 = decimal, datetime64 = fecha
print('Tipos actuales en ventas:')
print(ventas.dtypes)

# pd.to_datetime() → convierte una columna de strings al tipo fecha (datetime64)
# ventas['fecha'] → la columna a convertir; cada valor es un string como '15/01/2024'
# format='%d/%m/%Y' → le dice cómo leer el string: %d=día, %m=mes, %Y=año con 4 dígitos
# sin format= pandas intenta adivinar el formato y puede equivocarse o dar error
# ventas['fecha'] = → sobreescribe la columna con los valores ya convertidos a fecha
ventas['fecha'] = pd.to_datetime(ventas['fecha'], format='%d/%m/%Y')

# ventas['total'] → accede a la columna 'total'; cada valor es un string como '1799.98€'
# .str → activa el modo string: permite aplicar métodos de texto a cada valor de la columna
# .replace('€', '') → reemplaza '€' por '' (cadena vacía) en cada valor; resultado: '1799.98'
# regex=False → trata '€' como texto literal, no como expresión regular
# .astype(float) → convierte cada string resultante ('1799.98') a número decimal (1799.98)
# ventas['total'] = → sobreescribe la columna con los valores ya convertidos a número
ventas['total'] = ventas['total'].str.replace('€', '', regex=False).astype(float)

print('Tipos corregidos:')
print(ventas[['fecha', 'total']].dtypes)
print(ventas[['fecha', 'total']].head(3))

Tipos actuales en ventas:
venta_id       int64
producto_id    int64
fecha            str
cantidad       int64
total            str
cliente          str
dtype: object
Tipos corregidos:
fecha    datetime64[us]
total           float64
dtype: object
       fecha    total
0 2024-01-15  1799.98
1 2024-01-22   129.99
2 2024-01-30   839.97


---
## Concepto 5 — JOIN en pandas: `pd.merge()`

`pd.merge()` une dos DataFrames a través de una columna común. Es el equivalente al `INNER JOIN` de SQL.

```
pd.merge(df_izquierdo, df_derecho, on='columna_comun', how='inner')
```

- `on=` → columna con el mismo nombre en ambos DataFrames
- `how='inner'` → solo filas con coincidencia en ambos lados (es el valor por defecto)

**Ejercicio 5:** unir `ventas` con `productos` usando `producto_id`.

In [ ]:
# pd.merge() → une dos DataFrames por una columna que existe en ambos
# ventas → DataFrame izquierdo (el principal, el que queremos enriquecer con más datos)
# productos → DataFrame derecho (del que tomamos la información adicional)
# on='producto_id' → columna que actúa de puente; debe existir con ese nombre en ambos DataFrames
# how='inner' → devuelve solo las filas donde producto_id tiene coincidencia en los dos lados
# el resultado tiene todas las columnas de ventas más todas las columnas de productos
ventas_detalle = pd.merge(ventas, productos, on='producto_id', how='inner')

print('Ventas originales:   ', len(ventas))
print('Resultado del merge: ', len(ventas_detalle))
print('Primeras filas:')
# [['col1', 'col2', ...]] → selecciona solo esas columnas del resultado para una salida limpia
# .head() → muestra las primeras 5 filas por defecto; equivale a .head(5)
print(ventas_detalle[['venta_id', 'producto_id', 'nombre', 'categoria', 'total']].head())

Ventas originales:    50
Resultado del merge:  47
Primeras filas:
   venta_id  producto_id                   nombre    categoria    total
0         1          101       Laptop ProBook 450      Laptops  1799.98
1         2          103      Teclado Mecánico K3     Teclados   129.99
2         3          115  Auriculares AirPods Pro  Auriculares   839.97
3         4          107            Monitor 4K 27    Monitores   399.99
4         5          102     Monitor UltraWide 34    Monitores   899.98


---
## Concepto 6 — Tipos de merge: parámetro `how=`

| how | Qué devuelve | Equivalente SQL |
|-----|-------------|------------------|
| `inner` | Solo filas con coincidencia en ambos lados | INNER JOIN |
| `left` | Todas las filas del izquierdo, NaN si no hay coincidencia | LEFT JOIN |
| `right` | Todas las filas del derecho, NaN si no hay coincidencia | RIGHT JOIN |
| `outer` | Todas las filas de ambos lados | FULL OUTER JOIN |

El más común en análisis es `left`: conserva todos los registros del DataFrame principal aunque no tengan coincidencia en el otro.

**Ejercicio 6:** comparar cuántas filas devuelve `inner` vs `left` entre `ventas` y `productos`.

In [7]:
# hacemos los dos merges sobre los DataFrames originales (no el ya mergeado de ejercicio 5)
# how='inner' → descarta las ventas sin producto en catálogo
merge_inner = pd.merge(ventas, productos, on='producto_id', how='inner')

# how='left' → conserva TODAS las filas de ventas (DataFrame izquierdo)
# si una venta tiene un producto_id que no existe en productos,
# aparece de todas formas pero con NaN en todas las columnas que vienen de productos
merge_left  = pd.merge(ventas, productos, on='producto_id', how='left')

print('Inner:', len(merge_inner), 'filas')
print('Left: ', len(merge_left),  'filas')
# la diferencia entre left e inner es exactamente las ventas que inner descartó por no coincidir
print('Ventas sin producto en catálogo:', len(merge_left) - len(merge_inner))

# en el left merge, las ventas sin coincidencia tienen NaN en 'nombre' (columna de productos)
# merge_left['nombre'].isnull() → True donde 'nombre' es NaN → esas ventas no encontraron producto
# merge_left[...] → filtra y se queda solo con esas filas
# [['venta_id', 'producto_id']] → muestra solo las columnas útiles para identificarlas
huerfanas = merge_left[merge_left['nombre'].isnull()][['venta_id', 'producto_id']]
print('Ventas sin producto registrado:')
print(huerfanas)

Inner: 47 filas
Left:  50 filas
Ventas sin producto en catálogo: 3
Ventas sin producto registrado:
    venta_id  producto_id
47        48          148
48        49          149
49        50          150


---
## Concepto 7 — Verificar el resultado del merge

Después de un merge siempre hay que responder tres preguntas:

1. **¿Se perdieron filas?** Si el resultado tiene menos filas que el DataFrame izquierdo, hay claves sin coincidencia
2. **¿Se duplicaron filas?** Si tiene más filas, hay claves repetidas en el DataFrame derecho
3. **¿La clave sigue siendo única?** Verificar con `df['id'].duplicated().sum()`

Un merge 1:1 correcto devuelve exactamente las mismas filas que el DataFrame izquierdo.

**Ejercicio 7:** verificar la integridad del merge entre `ventas` y `productos`.

In [8]:
print('Ventas antes del merge: ', len(ventas))
print('Ventas después (inner): ', len(ventas_detalle))

# restamos filas del merge menos filas originales para saber si se ganó o perdió algo
# diff > 0 → el merge añadió filas (hay producto_ids repetidos en productos)
# diff < 0 → el merge perdió filas (hay ventas sin producto en catálogo)
# diff = 0 → relación perfecta 1:1
diff = len(ventas_detalle) - len(ventas)
if diff > 0:
    # cuando un producto_id aparece varias veces en productos,
    # cada venta que coincide con ese id se multiplica por el número de apariciones
    print('ALERTA: el merge añadió', diff, 'filas → hay duplicados en productos')
elif diff < 0:
    # abs() devuelve el valor absoluto: convierte el número negativo en positivo
    # ejemplo: abs(-3) devuelve 3
    print('INFO: se perdieron', abs(diff), 'ventas → no tienen producto registrado')
else:
    print('OK: merge 1:1, sin pérdidas ni duplicaciones')

# verificación final: comprobamos que venta_id sigue siendo único en el resultado
# ventas_detalle['venta_id'] → Serie con todos los ids de venta del resultado
# .duplicated() → True donde ese id ya apareció antes en la Serie
# .sum() → si es 0, cada venta_id es único y el merge fue correcto
duplicados_id = ventas_detalle['venta_id'].duplicated().sum()
print('venta_ids duplicados post-merge:', duplicados_id)

Ventas antes del merge:  50
Ventas después (inner):  47
INFO: se perdieron 3 ventas → no tienen producto registrado
venta_ids duplicados post-merge: 0
